# Демо: работа с пайплайном из ноутбукаНоутбук больше не содержит логики — только вызовы пакета `forexmodel`.Отлаживать и менять код удобнее в `.py`-модулях, а здесь смотреть результаты.Структура и описание изменений — в `README.md`.

In [ ]:
%load_ext autoreload%autoreload 2import sysfrom pathlib import PathROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))from forexmodel import load_config, setup_loggingfrom forexmodel.pipelines import build_dataset, run_backtest, run_trainingsetup_logging()cfg = load_config(ROOT / "configs" / "default.yaml")cfg.data.csv_path = str(ROOT / "data" / "raw" / "LKOH_1m_15-26.csv")  # свой путьcfg

## 1. Данные, признаки, разметкаПризнаки считаются на непрерывном ряде, выборки режутся после — и проверяются на непересечение.

In [ ]:
ds = build_dataset(cfg)print(len(ds.features), "признаков")for name, part in ds.splits.items():    print(f"{name:6} {len(part):6} баров")ds.train[["time", "close", "label"]].tail()

In [ ]:
# распределение классов: при асимметричных барьерах класс 1 должен быть заметным —# именно он забирает бары «на излёте»ds.train["label"].value_counts(normalize=True).sort_index()

## 2. ОбучениеCatBoost + CNN-BiLSTM + мета-модель. Артефакты падают в `artifacts/<run_name>/`.

In [ ]:
trained = run_training(cfg, dataset=ds)trained.metrics["primary"]

## 3. Бэктест`polar_edge` в метриках выше — главный индикатор: если он около нуля,преимущества у модели нет и настраивать симуляцию бессмысленно.

In [ ]:
result = run_backtest(    cfg,    split="sim",    dataset=ds,    primary=trained.primary,    nn_model=trained.nn,    meta_model=trained.meta,)result.report

## 4. Диагностика поздних входовЕсли winrate падает с ростом `extension_atr` — входы систематически поздние.

In [ ]:
result.diagnostics["late_entry"]

In [ ]:
result.diagnostics["exit_reasons"]

## 5. График сделок

In [ ]:
from forexmodel.viz import create_trading_chartcreate_trading_chart(    result.signals,    result.trades,    output_file=cfg.reports_path() / "chart_sim.html",    trend_col=cfg.simulation.trend_col,)

## 6. Перебор параметров

In [ ]:
from forexmodel.pipelines import run_sweepsweep = run_sweep(    cfg,    {"labeling.tp_atr": [1.25, 1.5, 2.0], "simulation.trail_atr": [1.0, 1.5]},    output_csv=cfg.reports_path() / "sweep_results.csv",)sweep.head(10)